# Xneural VAR: fair Granger-causality benchmarks

This notebook is generated from the completed experiment and contains only the reproducible protocol, result loading, final tables, and figures. No failed trial cells are included.


## Decision rules

- Xneural VAR and NGC/cMLP: an edge is present iff its proximal group is exactly nonzero. AUROC/AUPRC use the regularization path, not a post-hoc weight threshold.
- GVAR: continuous coefficient strength for AUROC/AUPRC and the original time-reversal stability rule for a binary graph.
- Linear VAR: joint F-test over all fitted lags and Benjamini-Hochberg FDR control at q=0.05.
- Unified comparison excludes diagonal self-edges; all-edge Lorenz metrics are retained in the JSON for Neural-GC comparability.


In [1]:
PROTOCOL = {'primary_claims': ['signed effective-coefficient interpretability', 'post-hoc-threshold-free exact sparsity for Xneural VAR and NGC/cMLP', 'Granger graph recovery on linear VAR and Neural-GC Lorenz-96 settings'], 'lorenz_reference': 'Tank et al. Neural Granger Causality, published TPAMI version; T=500 condition', 'lorenz_config': {'p': 20, 'order': 5, 'T': 500, 'forcing_values': [10.0, 40.0], 'delta_t': 0.05, 'observation_noise_sd': 0.1, 'burn_in': 1000, 'data_seed': 0, 'calibration_seed': 1000, 'evaluation_seeds': [0, 1, 2, 3, 4], 'hidden_layer_size': 100, 'num_hidden_layers': 1, 'max_epochs': 150, 'batch_size': 10000, 'ista_learning_rate': 0.05, 'gvar_learning_rate': 0.001, 'ridge_lambda': 0.01, 'lambda_smooth_xneural': 0.01, 'coefficient_weight_decay': 0.0001}, 'linear_config': {'p': 10, 'true_order': 1, 'fit_order': 5, 'T': 300, 'burn_in': 500, 'noise_sd': 0.5, 'innovation_correlation': 0.15, 'calibration_seed': 1000, 'evaluation_seeds': [0, 1, 2, 3, 4], 'hidden_layer_size': 32, 'num_hidden_layers': 1, 'max_epochs': 200, 'batch_size': 10000, 'ista_learning_rate': 0.05, 'gvar_learning_rate': 0.002, 'ridge_lambda': 0.01, 'lambda_smooth_xneural': 0.0, 'coefficient_weight_decay': 0.0001, 'fdr_level': 0.05}, 'unified_evaluation': 'off-diagonal target-source pairs', 'xneural_ngc_binary_rule': 'group norm > 0 exactly; no post-hoc threshold and no top-k', 'linear_var_binary_rule': 'joint all-lag F-test with Benjamini-Hochberg FDR q=0.05', 'gvar_binary_rule': 'time-reversal stability-based quantile selection, Q=20', 'calibration': 'one separate calibration seed; five evaluation seeds are not used for hyperparameter selection'}
PROTOCOL


{
  "primary_claims": [
    "signed effective-coefficient interpretability",
    "post-hoc-threshold-free exact sparsity for Xneural VAR and NGC/cMLP",
    "Granger graph recovery on linear VAR and Neural-GC Lorenz-96 settings"
  ],
  "lorenz_reference": "Tank et al. Neural Granger Causality, published TPAMI version; T=500 condition",
  "lorenz_config": {
    "p": 20,
    "order": 5,
    "T": 500,
    "forcing_values": [
      10.0,
      40.0
    ],
    "delta_t": 0.05,
    "observation_noise_sd": 0.1,
    "burn_in": 1000,
    "data_seed": 0,
    "calibration_seed": 1000,
    "evaluation_seeds": [
      0,
      1,
      2,
      3,
      4
    ],
    "hidden_layer_size": 100,
    "num_hidden_layers": 1,
    "max_epochs": 150,
    "batch_size": 10000,
    "ista_learning_rate": 0.05,
    "gvar_learning_rate": 0.001,
    "ridge_lambda": 0.01,
    "lambda_smooth_xneural": 0.01,
    "coefficient_weight_decay": 0.0001
  },
  "linear_config": {
    "p": 10,
    "true_order": 1,
    "fit_ord

In [2]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / 'experiments').exists():
    ROOT = ROOT.parent
RESULT_PATH = ROOT / 'experiments' / 'benchmark_results.json'
RESULTS = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
print('loaded:', RESULT_PATH)


loaded: experiments/benchmark_results.json


## Linear VAR(1)


In [3]:
condition = next(c for c in RESULTS['conditions'] if c['name'] == 'Linear VAR(1)')
print(condition['selected_hyperparameters'])


{'xneural_lambda': 0.12, 'cmlp_lambda': 0.4, 'gvar_lambda': 0.05, 'gvar_gamma': 0.02, 'linear_var_fdr_q': 0.05}


In [4]:
# Mean metrics across the five predeclared evaluation seeds.
from run_all_benchmarks import format_method_table
print(format_method_table(condition))


method                   AUROC  AUPRC  F1     BA     edges  sign 
-----------------------  -----  -----  -----  -----  -----  -----
Xneural VAR              0.805  0.565  0.629  0.784  27.8   0.750
NGC (cMLP)               0.976  0.924  0.866  0.911  19.8   --   
GVAR                     0.570  0.286  0.329  0.524  52.4   0.930
linear VAR (F-test, BH)  0.951  0.896  0.752  0.811  14.0   1.000


![comparison](../figures/linear_var(1)_comparison.png)

![path](../figures/linear_var(1)_xneural_path.png)


## Lorenz-96 F=10


In [5]:
condition = next(c for c in RESULTS['conditions'] if c['name'] == 'Lorenz-96 F=10')
print(condition['selected_hyperparameters'])


{'xneural_lambda': 0.16, 'cmlp_lambda': 0.2, 'gvar_lambda': 3.0, 'gvar_gamma': 0.0, 'linear_var_fdr_q': 0.05}


In [6]:
# Mean metrics across the five predeclared evaluation seeds.
from run_all_benchmarks import format_method_table
print(format_method_table(condition))


method                   AUROC  AUPRC  F1     BA     edges  sign
-----------------------  -----  -----  -----  -----  -----  ----
Xneural VAR              0.681  0.254  0.415  0.681  118.2  --  
NGC (cMLP)               0.991  0.954  0.910  0.948  60.4   --  
GVAR                     0.578  0.220  0.235  0.532  102.2  --  
linear VAR (F-test, BH)  0.866  0.690  0.506  0.676  27.0   --  


![comparison](../figures/lorenz-96_f10_comparison.png)

![path](../figures/lorenz-96_f10_xneural_path.png)


## Lorenz-96 F=40


In [7]:
condition = next(c for c in RESULTS['conditions'] if c['name'] == 'Lorenz-96 F=40')
print(condition['selected_hyperparameters'])


{'xneural_lambda': 0.16, 'cmlp_lambda': 0.15, 'gvar_lambda': 3.0, 'gvar_gamma': 0.025, 'linear_var_fdr_q': 0.05}


In [8]:
# Mean metrics across the five predeclared evaluation seeds.
from run_all_benchmarks import format_method_table
print(format_method_table(condition))


method                   AUROC  AUPRC  F1     BA     edges  sign
-----------------------  -----  -----  -----  -----  -----  ----
Xneural VAR              0.840  0.514  0.685  0.840  75.0   --  
NGC (cMLP)               0.954  0.792  0.744  0.924  96.0   --  
GVAR                     0.641  0.238  0.284  0.551  235.6  --  
linear VAR (F-test, BH)  0.892  0.772  0.494  0.665  21.0   --  


![comparison](../figures/lorenz-96_f40_comparison.png)

![path](../figures/lorenz-96_f40_xneural_path.png)


## Lag-wise gates and sign distributions

![F10 gate](../figures/lorenz96_f10_gate_by_lag_benchmark.png)

![F40 gate](../figures/lorenz96_f40_gate_by_lag_benchmark.png)

![linear gate](../figures/linear_var_gate_by_lag_benchmark.png)

![signed coefficients](../figures/linear_var_signed_boxplots_benchmark.png)


## Full rerun

Run the command below from the repository root to recompute every fit. Cached fits are used when `--force` is omitted.


In [ ]:
# %run experiments/run_all_benchmarks.py --force --workers 3
